# Ebook AI – RAG test (Colab)
Runs the service's own code: PDFs in Cloudflare R2 -> Chroma index (local + R2) -> Gemini answers.

**Before running**, add these in Colab → 🔑 *Secrets* (enable notebook access):
`R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_BUCKET`, `GOOGLE_API_KEY`.

Use the **development** bucket: building the index publishes it.

In [ ]:
!pip -q install "boto3>=1.36" PyPDF2 langchain-core langchain-text-splitters langchain-chroma langchain-huggingface langchain-google-genai sentence-transformers

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    def _secret(name):
        try: return userdata.get(name)
        except Exception: return os.environ.get(name)
except ImportError:
    def _secret(name): return os.environ.get(name)

for name in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET", "GOOGLE_API_KEY"):
    os.environ[name] = _secret(name) or getpass(f"{name}: ")
os.environ["CHROMA_DIR"] = "/content/chroma_db_local"
print("Config OK, bucket =", os.environ["R2_BUCKET"])

## 1. Upload the service modules
Choose `rag_service.py`, `r2_storage.py` and `catalog.py` from the repository.

In [ ]:
from google.colab import files
files.upload()

import r2_storage, rag_service
print("books/:", r2_storage.list_files("books"))

## 2. Upload PDFs to `books/` (optional)

In [ ]:
for name in files.upload():  # choose one or more .pdf files
    r2_storage.upload_file(f"books/{name}", name)
    print("uploaded books/" + name)

## 3. Build and publish the index

In [ ]:
rag_service.generate_and_store_database_task()
data, _ = r2_storage.read_data()
print("version:", data["index"]["version"], "| chunks:", data["index"]["chunks"])

## 4. Verify: download the published copy into a fresh folder

In [ ]:
rag_service.download_index(data["index"], "/content/chroma_db_from_r2")  # checks every sha256
print("download OK")

## 5. Query with Gemini

In [ ]:
rag_service.query_database("What is Flutter?")